In [ ]:
from ultralytics import YOLO
import torch
import os
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
DATA_YAML_PATH = r"D:\VScodefiles\DeepLearningProject\sonar_dataset_10k_3k_3k.yaml"
MODEL_NAME = "yolo11s.pt"

IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 100
WORKERS = 8

PROJECT_DIR = "runs/detect"
RUN_NAME = "yolo_full_100_epochs"

In [ ]:
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------
# Load Model
# --------------------------------
model = YOLO(MODEL_NAME)

# --------------------------------
# Train (NO resume, clean run)
# --------------------------------
results = model.train(
    data=DATA_YAML_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    workers=WORKERS,
    device=0,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=False,        # force new clean run
    save=True,
    pretrained=True,
    optimizer="auto",
    lr0=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    box=7.5,
    cls=0.5,
    dfl=1.5,
)

print("Training Complete")

# --------------------------------
# Get correct run directory
# --------------------------------
RUN_DIR = results.save_dir
print("Run Directory:", RUN_DIR)

results_csv_path = os.path.join(RUN_DIR, "results.csv")

# --------------------------------
# Verify CSV
# --------------------------------
if not os.path.exists(results_csv_path):
    raise FileNotFoundError("results.csv not found!")

df = pd.read_csv(results_csv_path)

print("Total epochs logged:", len(df))
print("Columns:")
print(df.columns.tolist())

# --------------------------------
# Ensure correct columns exist
# --------------------------------
required_columns = [
    "epoch",
    "time",
    "train/box_loss",
    "train/cls_loss",
    "train/dfl_loss",
    "metrics/precision(B)",
    "metrics/recall(B)",
    "metrics/mAP50(B)",
    "metrics/mAP50-95(B)",
    "val/box_loss",
    "val/cls_loss",
    "val/dfl_loss",
    "lr/pg0",
    "lr/pg1",
    "lr/pg2",
]

missing = [col for col in required_columns if col not in df.columns]

if missing:
    print("WARNING: Missing columns:", missing)

# --------------------------------
# Plotting (FULL 1–100)
# --------------------------------
PLOT_DIR = os.path.join(RUN_DIR, "custom_plots")
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(fig, name):
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)

# 1️⃣ Loss Curves
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["train/box_loss"], label="Train Box")
ax.plot(df["epoch"], df["val/box_loss"], label="Val Box")
ax.plot(df["epoch"], df["train/cls_loss"], label="Train CLS")
ax.plot(df["epoch"], df["val/cls_loss"], label="Val CLS")
ax.plot(df["epoch"], df["train/dfl_loss"], label="Train DFL")
ax.plot(df["epoch"], df["val/dfl_loss"], label="Val DFL")
ax.set_title("Train vs Val Loss (All Epochs)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True)
save_plot(fig, "loss_curves.png")

# 2️⃣ mAP
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
ax.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
ax.set_title("mAP over Epochs")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP")
ax.legend()
ax.grid(True)
save_plot(fig, "map_curves.png")

# 3️⃣ Precision & Recall
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
ax.plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
ax.set_title("Precision & Recall over Epochs")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.legend()
ax.grid(True)
save_plot(fig, "precision_recall.png")

# 4️⃣ Learning Rate
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], df["lr/pg0"], label="lr/pg0")
ax.plot(df["epoch"], df["lr/pg1"], label="lr/pg1")
ax.plot(df["epoch"], df["lr/pg2"], label="lr/pg2")
ax.set_title("Learning Rate Schedule")
ax.set_xlabel("Epoch")
ax.set_ylabel("LR")
ax.legend()
ax.grid(True)
save_plot(fig, "learning_rate.png")

print("All plots generated for full 1–100 epochs.")
